# MAVKA / RUKOPYS — Qwen3-VL Classification (T4 low-VRAM, fixed 224×224)

Fixes the original T4 OOM: correctly caps Qwen3-VL `image_processor.size["longest_edge"]`, letterboxes crops to 224×224 without aspect distortion, checks actual visual patch count, and saves a checkpoint after step 1 and every 10 steps. Uses a **new Google Drive experiment directory**; prior run failed before its first checkpoint (5/350). The data split is inherited from the existing persistent project root only if you manually copy it; by default deterministic split is regenerated in this new root using the same metadata and seed.

**Important:** This is a conservative first-pass classifier resolution, potentially reducing classification accuracy on small text. After confirming VRAM headroom, try 256/320 pixels as a separate experiment on a larger GPU. Do not increase resolution during a resumed run. Full GPU execution and resume after interruption have not been validated in this environment.


## 1. Dependencies and persistent Drive storage

In [ ]:
%pip -q install "transformers>=4.57,<5" "accelerate>=1.10,<2" "peft>=0.17,<1" "bitsandbytes>=0.45" "huggingface_hub>=0.34,<2" scikit-learn matplotlib tqdm safetensors
from google.colab import drive
drive.mount('/content/drive')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Mounted at /content/drive


## 2. Configuration and device check

In [ ]:
import os, json, math, random, hashlib, gc, warnings
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from huggingface_hub import hf_hub_download, dataset_info

SEED = 42
HF_REPO = 'UkrainianCatholicUniversity/rukopys'  # Official Kaggle competition source
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
ROOT = Path('/content/drive/MyDrive/rukopys_qwen_classifier_t4_224_v1')  # New experiment; prevents incompatible resume
ROOT.mkdir(parents=True, exist_ok=True)
CACHE = Path('/content/rukopys_cache')
CACHE.mkdir(parents=True, exist_ok=True)
VAL_FRACTION = 0.20
MAX_TRAIN_PER_CLASS = {'handwritten': 2000, 'formula': 2000, 'annotation': 500,
                       'printed': 300, 'table': 140, 'image': 110, 'graph': 55}
MAX_VAL_PER_CLASS = 120
MAX_IMAGE_PIXELS = 224 * 224  # HARD limit for Qwen3-VL image preprocessing
IMAGE_CANVAS = 224  # fixed square with aspect-ratio-preserving letterboxing
MAX_VISUAL_PATCHES = 256  # hard abort if the image processor unexpectedly expands images
STEPS = 350
ACCUM_STEPS = 8
LR = 1e-4
SAVE_EVERY = 10  # plus an early checkpoint after first optimizer update
CLASSES = ['handwritten', 'printed', 'formula', 'table', 'annotation', 'image', 'graph']
CLASS_PROMPT = ('Classify the main content of this document crop. Return exactly one label: '
    'handwritten, printed, formula, table, annotation, image, graph. '
    'Use annotation for marginal notes, signatures or correction marks; formula for mathematical expressions; '
    'table for tabular content; graph for plotted charts; image for non-text illustrations. No explanation.')
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))
print('Drive:', ROOT)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

GPU: Tesla T4 | VRAM GiB: 14.6
Drive: /content/drive/MyDrive/rukopys_qwen_classifier_t4_224_v1


## 2a. Reliable data source — official dataset linked by Kaggle

The competition ZIP may contain only competition-specific files. The complete labeled data are maintained at [UkrainianCatholicUniversity/rukopys](https://huggingface.co/datasets/UkrainianCatholicUniversity/rukopys) with:

- `train/metadata.jsonl` — ground-truth annotations
- `train/images/<uuid>.jpg` — corresponding pages

This cell **checks the actual metadata schema** before proceeding. `kaggle_token` is **not needed** here. Only selected page images are downloaded later, so we don't need to unpack the entire dataset.

In [ ]:
# Resolve and pin the exact HF dataset revision once for reproducibility.
# The first run needs internet; subsequent runs can reuse cached HF files.
from huggingface_hub import HfApi
revision_file = ROOT / 'dataset_revision.txt'
if revision_file.exists():
    REVISION = revision_file.read_text().strip()
else:
    REVISION = dataset_info(HF_REPO).sha
    revision_file.write_text(REVISION)

try:
    meta_path = Path(hf_hub_download(
        repo_id=HF_REPO,
        filename='train/metadata.jsonl',
        repo_type='dataset',
        revision=REVISION,
    ))
except Exception as exc:
    raise RuntimeError('Failed to download official RUKOPYS metadata. Check Colab internet and Hugging Face access.') from exc

print('Source:', HF_REPO)
print('Revision:', REVISION)
print('Metadata:', meta_path)
print('Size (MiB):', round(meta_path.stat().st_size / 2**20, 2))

with meta_path.open(encoding='utf-8') as f:
    example_record = json.loads(next(line for line in f if line.strip()))
assert {'file_name','regions'}.issubset(example_record), f'Unexpected metadata schema: {list(example_record)}'
assert str(example_record['file_name']).startswith('images/'), f'Unexpected page path: {example_record["file_name"]}'
assert isinstance(example_record['regions'], list), 'Expected list of regions'
print('First page:', example_record['file_name'])
print('First record has', len(example_record['regions']), 'regions')
print('First region:', example_record['regions'][0] if example_record['regions'] else '<empty>')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


metadata.jsonl: 0.00B [00:00, ?B/s]

Source: UkrainianCatholicUniversity/rukopys
Revision: 8217bf9a5189f5ce919de99e32e5db7bab37fc17
Metadata: /root/.cache/huggingface/hub/datasets--UkrainianCatholicUniversity--rukopys/snapshots/8217bf9a5189f5ce919de99e32e5db7bab37fc17/train/metadata.jsonl
Size (MiB): 4.27
First page: images/09ba34df-2665-452c-9ef4-6998a5e7944c.jpg
First record has 23 regions
First region: {'bbox': [1213, 350, 2052, 530], 'type': 'handwritten', 'language': 'uk', 'legibility': 'legible', 'text': 'Магія голосу.'}


## 3. Human-annotated metadata and reproducible page-disjoint split

In [ ]:
# Use the previously validated HF metadata; do not fetch the Kaggle archive.
assert meta_path.is_file()
records = [json.loads(line) for line in meta_path.read_text(encoding='utf8').splitlines() if line.strip()]
assert records and all('file_name' in r and 'regions' in r for r in records)
assert len({r['file_name'] for r in records}) == len(records), 'Duplicate page filenames: adjust page identity!'
meta_sha = hashlib.sha256(meta_path.read_bytes()).hexdigest()
split_path = ROOT / 'split.json'
if split_path.exists():
    split = json.loads(split_path.read_text())
    assert split['metadata_sha256'] == meta_sha, 'Metadata changed; use a NEW experiment directory.'
else:
    train_pages, val_pages = [], []
    by_source = defaultdict(list)
    for rec in records: by_source[rec.get('source', 'unknown')].append(rec['file_name'])
    for source in sorted(by_source):
        names = sorted(by_source[source], key=lambda x: hashlib.sha256(f'{SEED}|{x}'.encode()).hexdigest())
        n = max(1, math.ceil(len(names) * VAL_FRACTION)) if len(names) > 1 else 0
        n = min(n, len(names)-1)
        val_pages.extend(names[:n]); train_pages.extend(names[n:])
    split = {'metadata_sha256': meta_sha, 'dataset_revision': REVISION, 'seed': SEED,
             'train': sorted(train_pages), 'validation': sorted(val_pages),
             'limitation': 'Page-disjoint only; writers or repeated text may overlap.'}
    split_path.write_text(json.dumps(split, indent=2, ensure_ascii=False))
train_set, val_set = set(split['train']), set(split['validation'])
assert not (train_set & val_set)
assert train_set | val_set == {r['file_name'] for r in records}
print('Pages:', len(records), 'train:', len(train_set), 'validation:', len(val_set))
print('Revision:', REVISION, 'metadata sha256:', meta_sha[:16])

Pages: 1330 train: 1062 validation: 268
Revision: 8217bf9a5189f5ce919de99e32e5db7bab37fc17 metadata sha256: 5f6ece6e2c10ff68


## 4. Region-level examples and class distribution

In [ ]:
all_rows = []
for page in records:
    page_id = page['file_name']
    for idx, reg in enumerate(page['regions']):
        typ = reg.get('type')
        bbox = reg.get('bbox')
        if typ not in CLASSES or not isinstance(bbox, (list,tuple)) or len(bbox)!=4: continue
        try:
            box = [float(v) for v in bbox]
            if not all(math.isfinite(v) for v in box) or box[2]<=box[0] or box[3]<=box[1]: continue
        except (TypeError, ValueError): continue
        all_rows.append({'page':page_id,'idx':idx,'type':typ,'bbox':box,
                         'split':'train' if page_id in train_set else 'validation'})
frame = pd.DataFrame(all_rows)
print(pd.crosstab(frame['type'], frame['split']).reindex(CLASSES).fillna(0).astype(int))
assert set(CLASSES).issubset(set(frame.query("split == 'train'")['type'])), 'Some classes absent in training'
print('Region rows:', len(frame))

split        train  validation
type                          
handwritten  17001        4574
printed        241          67
formula       2415         477
table          122          20
annotation     421         133
image           75          44
graph           47          12
Region rows: 25649


## 5. Select a deterministic pilot subset **by class**, without leaking validation

In [ ]:
def choose_subset(data, caps, salt):
    selected=[]
    for label in CLASSES:
        group=data[data.type==label].copy()
        group['sort_key']=[hashlib.sha256(f'{SEED}|{salt}|{p}|{i}'.encode()).hexdigest()
                           for p,i in zip(group.page,group.idx)]
        selected.append(group.sort_values('sort_key').head(caps[label]).drop(columns='sort_key'))
    return pd.concat(selected, ignore_index=True).to_dict('records')
train_rows=choose_subset(frame[frame.split=='train'], MAX_TRAIN_PER_CLASS, 'training')
val_rows=choose_subset(frame[frame.split=='validation'], {k:MAX_VAL_PER_CLASS for k in CLASSES}, 'validation')
print('Train subset:', Counter(x['type'] for x in train_rows))
print('Validation subset:', Counter(x['type'] for x in val_rows))
assert set(x['page'] for x in train_rows).isdisjoint(x['page'] for x in val_rows)
assert len(val_rows)>0
(ROOT/'selected_manifest.json').write_text(json.dumps({'revision':REVISION,'train':train_rows,'validation':val_rows},ensure_ascii=False))

Train subset: Counter({'handwritten': 2000, 'formula': 2000, 'annotation': 421, 'printed': 241, 'table': 122, 'image': 75, 'graph': 47})
Validation subset: Counter({'handwritten': 120, 'formula': 120, 'annotation': 120, 'printed': 67, 'image': 44, 'table': 20, 'graph': 12})


813897

## 6. Download selected pages and create crops

Pages are downloaded on demand from the pinned official dataset revision (`train/images/*.jpg`). Re-runs skip already cached crops. Keep the Colab runtime alive while cropping; after a complete runtime reset, crops in `/content` are recreated. The source metadata and split are pinned for reproducibility.

In [ ]:
rows_by_page = defaultdict(list)
for row in train_rows + val_rows: rows_by_page[row['page']].append(row)

meta_lookup = {r['file_name']:r for r in records}
def page_path(page):
    # Metadata file_name is e.g. images/<uuid>.jpg, NOT an absolute path.
    relative = Path(page)
    if relative.is_absolute() or '..' in relative.parts or relative.parts[0] != 'images':
        raise ValueError(f'Unexpected page filename: {page!r}')
    return Path(hf_hub_download(
        repo_id=HF_REPO,
        filename='train/' + relative.as_posix(),
        repo_type='dataset', revision=REVISION))

def load_page(path, meta):
    with Image.open(path) as orig:
        img = orig.convert('RGB')
        w = meta.get('image_width'); h = meta.get('image_height')
        if w and h and img.size != (w,h):
            alt = ImageOps.exif_transpose(orig).convert('RGB')
            if alt.size == (w,h): img = alt
        return img

def crop_path(row):
    key = hashlib.sha256(f"{row['page']}|{row['idx']}".encode()).hexdigest()[:24]
    return CACHE/'crops'/row['split']/(key+'.jpg')

failed=[]
for page, page_rows in tqdm(rows_by_page.items(),desc='Pages'):
    if all(crop_path(row).is_file() for row in page_rows): continue
    try:
        img=load_page(page_path(page),meta_lookup[page]); w,h=img.size
        for row in page_rows:
            out=crop_path(row); out.parent.mkdir(parents=True,exist_ok=True)
            if out.is_file():continue
            x1,y1,x2,y2=row['bbox']
            box=(max(0,math.floor(x1)),max(0,math.floor(y1)),min(w,math.ceil(x2)),min(h,math.ceil(y2)))
            if box[2]<=box[0] or box[3]<=box[1]:
                failed.append((page,row['idx'],'invalid bounds')); continue
            img.crop(box).save(out,quality=95)
    except Exception as exc:
        failed.append((page, 'page', str(exc)))
train_rows = [r for r in train_rows if crop_path(r).is_file()]
val_rows = [r for r in val_rows if crop_path(r).is_file()]
print('Prepared train/validation:',len(train_rows),len(val_rows),'failed:',len(failed))
if failed:
    print('First failures:', failed[:15])
    if len(failed) > 0.05 * len(rows_by_page):
        raise RuntimeError('Too many page downloads or crop failures (>5%). See failures above; stop before training.')
print('Missing by class:',{label:sum(r['type']==label for r in train_rows)==0 for label in CLASSES})
assert all(any(r['type']==label for r in train_rows) for label in CLASSES)
assert val_rows
if failed: print('Example issues:',failed[:10])

Pages:   0%|          | 0/1156 [00:00<?, ?it/s]

train/images/381c13d7-a00d-551d-84c4-8ef(…):   0%|          | 0.00/4.64M [00:00<?, ?B/s]

train/images/9421d963-534f-5f97-b799-025(…):   0%|          | 0.00/2.92M [00:00<?, ?B/s]

train/images/3a2cb6f7-7295-5ef3-9316-674(…):   0%|          | 0.00/4.06M [00:00<?, ?B/s]

train/images/d814fc6f-3862-5da8-b3c7-fba(…):   0%|          | 0.00/3.43M [00:00<?, ?B/s]

train/images/c919f536-1f47-59e1-ac4e-74d(…):   0%|          | 0.00/2.69M [00:00<?, ?B/s]

... (1,150 outputs left out)

train/images/e19d3af0-771e-493c-916b-8f3(…):   0%|          | 0.00/300k [00:00<?, ?B/s]

train/images/33066d82-e7bc-4753-8bb9-30e(…):   0%|          | 0.00/2.06M [00:00<?, ?B/s]

train/images/ec12ba43-51b1-4ac0-a52a-b3a(…):   0%|          | 0.00/243k [00:00<?, ?B/s]

Prepared train/validation: 4906 503 failed: 0
Missing by class: {'handwritten': False, 'printed': False, 'formula': False, 'table': False, 'annotation': False, 'image': False, 'graph': False}


## 7. Load quantized Qwen and attach a classification-only LoRA adapter

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=DTYPE,bnb_4bit_use_double_quant=True)
processor=AutoProcessor.from_pretrained(MODEL_ID)
# Qwen3-VL uses image_processor.size["longest_edge"] as its effective max_pixels.
# Merely setting an attribute called `max_pixels` did NOT reliably enforce the limit.
ip = processor.image_processor
if not isinstance(getattr(ip, 'size', None), dict) or 'longest_edge' not in ip.size:
    raise RuntimeError(f'Unexpected image processor size config: {getattr(ip, "size", None)!r}. Inspect this before training.')
ip.size['longest_edge'] = MAX_IMAGE_PIXELS
if 'shortest_edge' in ip.size:
    ip.size['shortest_edge'] = min(ip.size['shortest_edge'], MAX_IMAGE_PIXELS)
if hasattr(ip, 'max_pixels'):
    ip.max_pixels = MAX_IMAGE_PIXELS
print('ENFORCED image processor size:', ip.size)
model=Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,quantization_config=quant,device_map='auto',torch_dtype=DTYPE)
model.config.use_cache=False
# Gradient checkpointing already enabled below. No torch.compile or cache on T4.
model=prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                      gradient_checkpointing_kwargs={'use_reentrant':False})
targets=[name for name,_ in model.named_modules() if 'language_model' in name
         and name.split('.')[-1] in ('q_proj','k_proj','v_proj','o_proj')]
assert targets, 'LoRA language attention modules not found: review model architecture.'
model=get_peft_model(model,LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,
                                    target_modules=targets,bias='none',task_type='CAUSAL_LM'))
model.print_trainable_parameters()
model.train()

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

... (8 outputs left out)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

trainable params: 11,796,480 || all params: 4,449,612,288 || trainable%: 0.2651


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3VLForConditionalGeneration(
      (model): Qwen3VLModel(
        (visual): Qwen3VLVisionModel(
          (patch_embed): Qwen3VLVisionPatchEmbed(
            (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1024)
          (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-23): 24 x Qwen3VLVisionBlock(
              (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
              (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True, bias=True)
              (attn): Qwen3VLVisionAttention(
                (qkv): Linear4bit(in_features=1024, out_features=3072, bias=True)
                (proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
              )
              (mlp): Qwen3VLVisionMLP(
                (linear_fc1): Linear4bit(in_features=1024, out_feature

## 8. Supervised example (assistant-only loss)

In [ ]:
from PIL import ImageOps

def fit_image_for_t4(image):
    # Create a small fixed-size input to bound the vision encoder's memory,
    # maintaining the original aspect ratio (no distortion).
    image = ImageOps.exif_transpose(image.convert('RGB'))
    scaled = image.copy()
    scaled.thumbnail((IMAGE_CANVAS, IMAGE_CANVAS), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (IMAGE_CANVAS, IMAGE_CANVAS), (255, 255, 255))
    canvas.paste(scaled, ((IMAGE_CANVAS-scaled.width)//2, (IMAGE_CANVAS-scaled.height)//2))
    return canvas

def assert_vision_budget(batch):
    grid=batch.get('image_grid_thw')
    if grid is None:
        raise RuntimeError('Expected image_grid_thw from Qwen3-VL processor; inspect outputs before training.')
    patch_count=int(grid.prod(dim=-1).sum().item())
    if patch_count > MAX_VISUAL_PATCHES:
        raise RuntimeError(f'IMAGE TOO LARGE: {patch_count} vision patches (cap {MAX_VISUAL_PATCHES}); grid={grid.tolist()}')
    return patch_count

def make_messages(image, answer=None):
    messages=[{'role':'user','content':[{'type':'image','image':image},
                                      {'type':'text','text':CLASS_PROMPT}]}]
    if answer is not None:
        messages.append({'role':'assistant','content':[{'type':'text','text':answer}]})
    return messages

def batch_for_row(row, training=True):
    with Image.open(crop_path(row)) as im: img=fit_image_for_t4(im)
    prefix=processor.apply_chat_template(make_messages(img),tokenize=True,
          add_generation_prompt=True,return_dict=True,return_tensors='pt')
    if not training:
        prefix.pop('token_type_ids',None)
        assert_vision_budget(prefix)
        return prefix.to('cuda')
    full=processor.apply_chat_template(make_messages(img,row['type']),tokenize=True,
          add_generation_prompt=False,return_dict=True,return_tensors='pt')
    n=prefix['input_ids'].shape[1]
    assert torch.equal(prefix['input_ids'],full['input_ids'][:,:n]),'Template prefix mismatch'
    labels=full['input_ids'].clone(); labels[:,:n]=-100
    labels[full['attention_mask']==0]=-100
    assert (labels!=-100).any()
    assert_vision_budget(full)
    full['labels']=labels
    full.pop('token_type_ids',None)
    return full.to('cuda')

sample=batch_for_row(train_rows[0]); print('Tokens:', sample['input_ids'].shape[1], 'supervised:', int((sample['labels']!=-100).sum()), 'visual patches:', assert_vision_budget(sample)); print('VRAM allocated GiB:', round(torch.cuda.memory_allocated()/2**30, 2))
del sample; torch.cuda.empty_cache()

Tokens: 128 supervised: 4 visual patches: 196
VRAM allocated GiB: 3.48


## 8a. Required memory smoke test before training


In [ ]:
# VRAM smoke test: one complete forward + backward on a representative crop.
# If this fails, do NOT start the 350-step training loop.
import time
model.train()
trainable=[p for p in model.parameters() if p.requires_grad]
probe_candidates=sorted(train_rows, key=lambda r: (r['type']!='table', r['type']!='formula'))
probe=probe_candidates[0]
probe_batch=batch_for_row(probe)
print('Smoke test class:', probe['type'], 'vision patches:', assert_vision_budget(probe_batch))
model.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
try:
    with torch.autocast('cuda', dtype=DTYPE):
        probe_loss=model(**probe_batch).loss
    probe_loss.backward()
    print('PASS: forward+backward. loss:', float(probe_loss.detach()),
          'peak VRAM GiB:', round(torch.cuda.max_memory_allocated()/2**30,2))
finally:
    model.zero_grad(set_to_none=True)
    del probe_batch
    if 'probe_loss' in globals(): del probe_loss
    gc.collect()
    torch.cuda.empty_cache()


Smoke test class: table vision patches: 196
PASS: forward+backward. loss: 6.250026702880859 peak VRAM GiB: 4.48


## 9. Train with resumable checkpoints (T4)

The earlier run died at 5/350 before the first save (25 steps), therefore **those 5 optimizer steps cannot be recovered**. This version saves a checkpoint after the first completed step, then every 10 steps. Uses its own Drive directory so checkpoints cannot be confused with the old image-resolution experiment. After a runtime restart, rerun setup and training cells in order; checkpoints remain in Drive.


In [ ]:
# Full resumable training: LoRA adapter + optimizer + AMP scaler + RNG + progress.
# Checkpoints are written to persistent Google Drive and marked COMPLETE only after all files exist.
from peft import set_peft_model_state_dict
from peft.utils.save_and_load import load_peft_weights
import shutil

by_class={cl:[r for r in train_rows if r['type']==cl] for cl in CLASSES}
assert all(by_class.values()), 'Every class must have at least one training row.'
optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR,weight_decay=0.0)
scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
CKPT_ROOT=ROOT/'checkpoints'; CKPT_ROOT.mkdir(parents=True,exist_ok=True)
RUN_CONFIG={'model':MODEL_ID,'seed':SEED,'classes':CLASSES,'lr':LR,
            'accum_steps':ACCUM_STEPS,'max_image_pixels':MAX_IMAGE_PIXELS,'image_canvas':IMAGE_CANVAS,'max_visual_patches':MAX_VISUAL_PATCHES,
            'train_ids':sorted([f"{r['page']}::{r['idx']}" for r in train_rows])}
config_hash=hashlib.sha256(json.dumps(RUN_CONFIG,sort_keys=True).encode()).hexdigest()

def latest_complete_checkpoint():
    checkpoints=[]
    for directory in CKPT_ROOT.glob('step_*'):
        if directory.is_dir() and (directory/'COMPLETE').exists() and (directory/'training_state.pt').exists():
            try: checkpoints.append((int(directory.name.split('_')[1]),directory))
            except (ValueError,IndexError): pass
    return max(checkpoints,default=(0,None))

def save_checkpoint(step,history):
    target=CKPT_ROOT/f'step_{step:06d}'
    staging=CKPT_ROOT/f'.incomplete_step_{step:06d}'
    if staging.exists(): shutil.rmtree(staging)
    staging.mkdir(parents=True)
    model.save_pretrained(staging,safe_serialization=True)
    processor.save_pretrained(staging)
    state={'step':step,'config_hash':config_hash,'optimizer':optimizer.state_dict(),
           'scaler':scaler.state_dict(),'history':history,
           'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
           'torch_rng':torch.get_rng_state(),
           'cuda_rng':torch.cuda.get_rng_state_all()}
    torch.save(state,staging/'training_state.pt')
    (staging/'COMPLETE').write_text(str(step))
    if target.exists(): shutil.rmtree(target)
    staging.rename(target)
    pd.DataFrame(history).to_csv(ROOT/'training_history.csv',index=False)
    print(f'Saved complete checkpoint: {target} | loss {history[-1]["loss"]:.4f}')

def restore_checkpoint(target):
    # Load only checkpoints created by this trusted notebook.
    state=torch.load(target/'training_state.pt',map_location='cpu',weights_only=False)
    if state['config_hash']!=config_hash:
        raise RuntimeError('Checkpoint config or training subset changed. Revert configuration, or use a new ROOT.')
    adapter_weights=load_peft_weights(str(target),device='cpu')
    set_peft_model_state_dict(model,adapter_weights,adapter_name='default')
    optimizer.load_state_dict(state['optimizer'])
    scaler.load_state_dict(state['scaler'])
    random.setstate(state['python_rng'])
    np.random.set_state(state['numpy_rng'])
    torch.set_rng_state(state['torch_rng'])
    torch.cuda.set_rng_state_all(state['cuda_rng'])
    return state['step'],state['history']

completed_step,ckpt=latest_complete_checkpoint()
if ckpt is not None:
    completed_step,history=restore_checkpoint(ckpt)
    print(f'Resuming from optimizer step {completed_step} (checkpoint {ckpt.name})')
else:
    completed_step,history=0,[]
    print('No complete checkpoint: starting a new run')

if completed_step>=STEPS:
    print(f'Run already reached {completed_step} steps; increase STEPS to continue.')
else:
    model.train(); optimizer.zero_grad(set_to_none=True)
    for step in tqdm(range(completed_step+1,STEPS+1),desc='Optimizer updates'):
        step_loss=0.0
        for micro in range(ACCUM_STEPS):
            label=CLASSES[random.randrange(len(CLASSES))]
            row=random.choice(by_class[label])
            batch=batch_for_row(row)
            with torch.autocast('cuda',dtype=DTYPE):
                loss=model(**batch).loss/ACCUM_STEPS
            scaler.scale(loss).backward()
            step_loss+=float(loss.detach())
            del batch,loss
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        history.append({'step':step,'loss':step_loss})
        if step<=3 or step%10==0:
            print(f'step={step} loss={step_loss:.4f} peak_VRAM_GiB={torch.cuda.max_memory_allocated()/2**30:.2f}')
        if step==1 or step%SAVE_EVERY==0 or step==STEPS:
            save_checkpoint(step,history)
    print('Training complete')


No complete checkpoint: starting a new run


Optimizer updates:   0%|          | 0/350 [00:00<?, ?it/s]

step=1 loss=2.9030 peak_VRAM_GiB=4.53
Saved complete checkpoint: /content/drive/MyDrive/rukopys_qwen_classifier_t4_224_v1/checkpoints/step_000001 | loss 2.9030
step=2 loss=2.6171 peak_VRAM_GiB=4.62
step=3 loss=1.2791 peak_VRAM_GiB=4.62
step=10 loss=0.3546 peak_VRAM_GiB=4.62
Saved complete checkpoint: /content/drive/MyDrive/rukopys_qwen_classifier_t4_224_v1/checkpoints/step_000010 | loss 0.3546
step=20 loss=0.2659 peak_VRAM_GiB=4.62
Saved complete checkpoint: /content/drive/MyDrive/rukopys_qwen_classifier_t4_224_v1/checkpoints/step_000020 | loss 0.2659
step=30 loss=0.0223 peak_VRAM_GiB=4.62
Saved complete checkpoint: /content/drive/MyDrive/rukopys_qwen_classifier_t4_224_v1/checkpoints/step_000030 | loss 0.0223
step=40 loss=0.0747 peak_VRAM_GiB=4.62
Saved complete checkpoint: /content/drive/MyDrive/rukopys_qwen_classifier_t4_224_v1/checkpoints/step_000040 | loss 0.0747
step=50 loss=0.1720 peak_VRAM_GiB=4.62
Saved complete checkpoint: /content/drive/MyDrive/rukopys_qwen_classifier_t4_224_

## 10. Validation: actual generated labels, macro-F1 and confusion matrix

Run only after training. This evaluates the currently loaded adapter; set `MAX_VAL_PER_CLASS` higher and rebuild crops for final reporting.

In [ ]:
def predict(row):
    model.eval()
    inputs=batch_for_row(row,training=False)
    with torch.inference_mode():
        generated=model.generate(**inputs,max_new_tokens=16,do_sample=False,
                                 pad_token_id=processor.tokenizer.eos_token_id)
    answer=processor.batch_decode(generated[:,inputs['input_ids'].shape[1]:],skip_special_tokens=True)[0].strip().lower()
    answer=answer.strip(' .`\n\t')
    del inputs,generated
    return answer if answer in CLASSES else '__invalid__'

pred=[]
for r in tqdm(val_rows,desc='Validation'):
    pred.append({'page':r['page'],'idx':r['idx'],'true':r['type'],'pred':predict(r)})
preds=pd.DataFrame(pred)
labels=CLASSES+(['__invalid__'] if '__invalid__' in set(preds.pred) else [])
print(classification_report(preds['true'],preds['pred'],labels=labels,zero_division=0))
macro=f1_score(preds['true'],preds['pred'],labels=CLASSES,average='macro',zero_division=0)
accuracy=accuracy_score(preds['true'],preds['pred'])
print('Macro-F1:',round(macro,5),'Accuracy:',round(accuracy,5))
print(pd.DataFrame(confusion_matrix(preds['true'],preds['pred'],labels=labels),index=labels,columns=labels))
preds.to_csv(ROOT/'validation_predictions.csv',index=False)
(ROOT/'metrics.json').write_text(json.dumps({'macro_f1':macro,'accuracy':accuracy,
                    'validation_size':len(preds),'labels':CLASSES,'model':MODEL_ID,
                    'dataset_revision':REVISION,'metadata_sha256':meta_sha},indent=2))
print('Metrics saved to',ROOT)

Validation:   0%|          | 0/503 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


              precision    recall  f1-score   support

 handwritten       0.96      0.74      0.84       120
     printed       0.71      0.93      0.81        67
     formula       0.96      0.97      0.96       120
       table       1.00      0.90      0.95        20
  annotation       0.82      0.96      0.88       120
       image       0.97      0.75      0.85        44
       graph       1.00      0.83      0.91        12

    accuracy                           0.88       503
   macro avg       0.92      0.87      0.88       503
weighted avg       0.90      0.88      0.88       503

Macro-F1: 0.88439 Accuracy: 0.88072
             handwritten  printed  formula  table  annotation  image  graph
handwritten           89       18        4      0           9      0      0
printed                1       62        0      0           3      1      0
formula                0        2      116      0           2      0      0
table                  1        0        1     18           0  

## 11. Next experiment and verification limits

After validating the classifier, train a **separate** formula/table transcription adapter. Freeze the same dataset revision and validation pages; compare base-vs-finetuned on identical examples. The competition's metric combines detection, classification and transcription, so classifier macro-F1 is an internal diagnostic, not the Kaggle score.

**Checks completed for this notebook:** JSON parsing, Python-cell compilation, source/path/schema handling tests with mocked HF metadata and images. **Not yet tested:** actual Colab runtime, Qwen GPU forward/backward or checkpoint recovery under a real disconnect. For the first execution, run cells in order, verify metadata and crops, then one training step before a long run.